In [11]:
from neo4j import GraphDatabase
import re

# Source file used by extract_structured_text
filename = "2408.13296v3.txt"

# Single Neo4j instance used only for KG relationships (triplets)
URI = "neo4j://localhost:7687"
NEO4J_AUTH = ("neo4j", "testpassword")

# Two separate vector storages for title retrieval
PARAGRAPH_VECTOR_STORE_PATH = "paragraph_title_vectors.json"
SECTION_VECTOR_STORE_PATH = "section_title_vectors.json"
EMBEDDING_MODEL = "mxbai-embed-large"

In [2]:
def extract_structured_text(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        # Split the text into blocks separated by at least one empty line
        raw_blocks = content.split('\n\n')
        
        structured_data = []
        
        for block in raw_blocks:
            # Clean the block and split into internal lines
            lines = [line.strip() for line in block.split('\n') if line.strip()]
            
            if not lines:
                continue
            
            # Apply your redefined rules
            if len(lines) == 1:
                structured_data.append({
                    "type": "Title",
                    "text": lines[0]
                })
            else:
                structured_data.append({
                    "type": "Paragraph",
                    "text": " ".join(lines)  # Join lines into a single string
                })
        
        return structured_data

    except FileNotFoundError:
        return "Error: File not found."

# Execution
data = extract_structured_text(filename)

# Displaying results
for item in data:
    print(f"[{item['type'].upper()}]: {item['text'][:70]}...")

[TITLE]: Chapter 1...
[TITLE]: Introduction...
[PARAGRAPH]: 1.1     Background of Large Language Models (LLMs) Large Language Mode...
[PARAGRAPH]: 1.2     Historical Development and Key Milestones Language models are ...
[PARAGRAPH]: 1.3     Evolution from Traditional NLP Models to State-of-the-Art LLMs...
[PARAGRAPH]: 1.3.1    Statistical Language Models (SLMs) Emerging in the 1990s, SLM...
[PARAGRAPH]: P (S) = P (ω1 , ω2 , ω3 , ω4 ) = P (I, am, very, happy)               ...
[PARAGRAPH]: P (I, am, very, happy) = P (I) · P (am | I) · P (very | I, am) · P (ha...
[PARAGRAPH]: 6 Figure 1.1: A chronological timeline showcasing the evolution of Lar...
[PARAGRAPH]: C(ω1 ω2 · · · ωi ) P (ωi | ω1 ω2 · · · ωi−1 ) =                       ...
[PARAGRAPH]: 1.3.2    Neural Language Models (NLMs) NLMs leverage neural networks t...
[PARAGRAPH]: 1.3.3    Pre-trained Language Models (PLMs) PLMs are initially trained...
[PARAGRAPH]: 1.3.4    Large Language Models (LLMs) LLMs like GPT-3, GPT-4, PaLM [10

In [4]:
import re

def stream_relationships(large_text):
    # This pattern specifically targets the Relationship Format:
    # It looks for: #Number [Subject] -> [Verb] -> [Object]
    # We use \s* to handle any inconsistent spacing around the arrows.
    rel_pattern = re.compile(r"#(\d+)\s+(.*?)\s*->\s*(.*?)\s*->\s*(.*)")

    # Splitting by newline to simulate line-by-line reading
    for line in large_text.splitlines():
        clean_line = line.strip()
        
        # We only care about lines that match our specific relationship pattern
        match = rel_pattern.search(clean_line)
        if match:
            yield {
                "id": int(match.group(1)),
                "subject": match.group(2).strip(),
                "verb": match.group(3).replace(" ",""),
                "object": match.group(4).strip()
            }


In [5]:
import ollama
import json

def self_discover_and_extract(title, paragraphs, entity_types, predicates):
    sample_text = f"Title: {title}\n\n" + "\n".join(paragraphs[:3])

    discovery_prompt = f"""Perform Named Entity Recognition (NER) and extract knowledge graph triplets from the text. NER identifies named entities of given entity types, and triple extraction identifies relationships between entities using specified predicates.

        **Entity Types:**
        {json.dumps(entity_types)}

        **Predicates:**
        {json.dumps(predicates)}

        **Text:**
        {sample_text}

        **Example Output:**
            "entities": [{{ "text": "Google", "type": "ORGANIZATION" }}],
            "triplets": [{{ "subject": "Google", "predicate": "founded_in", "object": "USA" }}]
        """

    discovery_prompt=f""" 
                        Prompt for extracting entities: Extract key entities from the given
                        text. Extracted entities are nouns, verbs, or adjectives,
                        particularly regarding sentiment. This is for an extraction
                        task, please be thorough and accurate to the reference text.

                        Prompt for extracting relations: Extract subject-predicate-object
                        triples from the assistant message. A predicate (1-3
                        words) defines the relationship between the subject and
                        object. Relationship may be fact or sentiment based on
                        assistant’s message. Subject and object are entities.
                        Entities provided are from the assistant message and
                        prior conversation history, though you may not need all of
                        them. This is for an extraction task, please be thorough,
                        accurate, and faithful to the reference text

                        In the end represent all the relationships with this format, do not add text:
                        
                        *Relationship Format:*

                        #[Number] Entity -> Predicate -> Entity

                        {sample_text}
                        """

    # Remove the redundant .format() call — the f-string already handled substitution
    discovery_response = ollama.chat(model='llama3:8b', messages=[
        {'role': 'user', 'content': discovery_prompt}
    ])
    return discovery_response['message']['content']

# Usage
# schema, triplets = self_discover_and_extract(my_title, my_paragraphs)for p in range(len(data)-1):

for p in range(len(data)-1):
    current_val =data[p]
    next_val = data[p+1]

    if current_val['type'] == 'Title' and next_val['type'] == 'Paragraph':
        my_title = current_val['text']
        my_paragraphs = [next_val['text']]

        

        entity_types = ["Person", "Organization", "Object", "Subject"]
        predicates = ["related_to", "located_in", "participated_in"]

        data_n = self_discover_and_extract(my_title, my_paragraphs, entity_types, predicates)
        extracted_triplets = list(stream_relationships(data_n))
        # 3. Print the actual parsed dictionaries
        print(f"--- Results for: {my_title} ---")
        for triplet in extracted_triplets:
            print(triplet)

--- Results for: Introduction ---
{'id': 1, 'subject': 'LLMs', 'verb': 'represent', 'object': 'computational systems'}
{'id': 2, 'subject': 'LLMs', 'verb': 'understand', 'object': 'human language'}
{'id': 3, 'subject': 'N-gram models', 'verb': 'leverage', 'object': 'Transformer architectures'}
{'id': 4, 'subject': 'GPT-3 and GPT-4', 'verb': 'leverage', 'object': 'self-attention mechanism within Transformer architectures'}
{'id': 5, 'subject': 'LLMs', 'verb': 'generate', 'object': 'coherent text from prompts'}
{'id': 6, 'subject': 'RLHF', 'verb': 'refine', 'object': 'models using human responses'}
{'id': 7, 'subject': 'Techniques (prompt engineering, question-answering, and conversational interactions)', 'verb': 'advance', 'object': 'field of natural language processing (NLP)'}
--- Results for: Stage 1: Data Preparation ---
--- Results for: 4.1     Steps Involved in Model Initialisation ---
{'id': 1, 'subject': 'Model', 'verb': 'IsInvolvedIn', 'object': 'Steps'}
{'id': 2, 'subject': 'St

In [6]:
def upload_triplets(driver, triplets, paragraph_name):
    with driver.session() as session:
        for triplet in triplets:
            rel_type = re.sub(r'[^a-zA-Z0-9_]', '', triplet['verb'].replace(" ", "_").upper())
            if not rel_type:
                rel_type = "RELATED_TO"

            # Keep relation and add paragraph context links for both entities.
            query = (
                f"MERGE (s:Entity {{name: $sub}}) "
                f"MERGE (o:Entity {{name: $obj}}) "
                f"MERGE (p:Paragraph {{name: $paragraph_name}}) "
                f"MERGE (s)-[:{rel_type}]->(o) "
                f"MERGE (s)-[:MENTIONED_IN]->(p) "
                f"MERGE (o)-[:MENTIONED_IN]->(p)"
            )
            session.run(
                query,
                sub=triplet['subject'],
                obj=triplet['object'],
                paragraph_name=paragraph_name,
            )

In [7]:
def upload_triplets_APOC(driver, triplets, paragraph_name):
    with driver.session() as session:
        for triplet in triplets:
            # Clean the verb to make it a valid Neo4j Relationship Type
            rel_type = re.sub(r'[^a-zA-Z0-9_]', '', triplet['verb'].replace(" ", "_").upper())
            if not rel_type:
                rel_type = "RELATED_TO"

            query = """
            MERGE (s:Entity {name: $sub})
            MERGE (o:Entity {name: $obj})
            MERGE (p:Paragraph {name: $paragraph_name})
            WITH s, o, p
            CALL apoc.create.relationship(s, $rel, {}, o) YIELD rel
            MERGE (s)-[:MENTIONED_IN]->(p)
            MERGE (o)-[:MENTIONED_IN]->(p)
            RETURN rel
            """
            session.run(
                query,
                sub=triplet['subject'],
                obj=triplet['object'],
                paragraph_name=paragraph_name,
                rel=rel_type,
            )

In [8]:
# Integration with Ollama loop + paragraph-context links
with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
    driver.verify_connectivity()

    for p in range(len(data) - 1):
        current_val = data[p]
        next_val = data[p + 1]

        if current_val['type'] == 'Title' and next_val['type'] == 'Paragraph':
            my_title = current_val['text']
            my_paragraphs = [next_val['text']]

            entity_types = ["Person", "Organization", "Object", "Subject"]
            predicates = ["related_to", "located_in", "participated_in"]

            data_n = self_discover_and_extract(my_title, my_paragraphs, entity_types, predicates)
            triplets = list(stream_relationships(data_n))

            if triplets:
                upload_triplets(driver, triplets, my_title)
                print(f"Uploaded {len(triplets)} relationships for paragraph '{my_title}'.")
            else:
                print(f"No relationships parsed for paragraph '{my_title}'; skipping Neo4j upload.")

Uploaded 9 relationships for paragraph 'Introduction'.
No relationships parsed for paragraph 'Stage 1: Data Preparation'; skipping Neo4j upload.
No relationships parsed for paragraph '4.1     Steps Involved in Model Initialisation'; skipping Neo4j upload.
Uploaded 12 relationships for paragraph 'Stage 3: Training Setup'.
No relationships parsed for paragraph '6.4.2    Comparison between HFT and LoRA'; skipping Neo4j upload.
Uploaded 5 relationships for paragraph 'Table 6.3: Comparative Analysis of Half Fine-Tuning (HFT) and Low-Rank Adaptation (LoRA).'.
Uploaded 4 relationships for paragraph '• Prompt'.
Uploaded 4 relationships for paragraph '• Supervised Fine-tuning Loss (SFT):'.
No relationships parsed for paragraph 'where yik is a binary indicator for the i-th token in the vocabulary, and pki is its predicted probability.'; skipping Neo4j upload.
Uploaded 4 relationships for paragraph 'Stage 5: Evaluation and Validation'.
No relationships parsed for paragraph 'Other tunable paramete

In [12]:
# Separate embeddings for section titles and paragraph titles + separate retrieval
from typing import List, Tuple
import json
import math
import ollama

def classify_title_kind(title: str) -> str:
    """Heuristic split between section titles and paragraph titles."""
    t = title.strip()
    if re.match(r"^(section\s+)?\d+(\.\d+)*([:)\-\s]|$)", t, flags=re.IGNORECASE):
        return "section"
    return "paragraph"

def split_titles_for_embedding(structured_data: List[dict]) -> Tuple[List[str], List[str]]:
    section_titles = []
    paragraph_titles = []
    for item in structured_data:
        if item.get("type") != "Title":
            continue
        title = item.get("text", "").strip()
        if not title:
            continue
        if classify_title_kind(title) == "section":
            section_titles.append(title)
        else:
            paragraph_titles.append(title)
    return paragraph_titles, section_titles

def build_embedding_records(titles: List[str], model: str) -> List[dict]:
    records = []
    for title in titles:
        emb = ollama.embeddings(model=model, prompt=title)["embedding"]
        records.append({"name": title, "embedding": emb})
    return records

def save_vector_store(path: str, records: List[dict]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f)

def load_vector_store(path: str) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def retrieve_similar_titles(vector_store_path: str, query_text: str, model: str, top_k: int = 5):
    records = load_vector_store(vector_store_path)
    query_embedding = ollama.embeddings(model=model, prompt=query_text)["embedding"]

    scored = []
    for row in records:
        score = cosine_similarity(query_embedding, row["embedding"])
        scored.append({"name": row["name"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

# Build two separate vector storages
paragraph_titles, section_titles = split_titles_for_embedding(data)
print(f"Paragraph titles: {len(paragraph_titles)}")
print(f"Section titles: {len(section_titles)}")

paragraph_records = build_embedding_records(paragraph_titles, EMBEDDING_MODEL)
section_records = build_embedding_records(section_titles, EMBEDDING_MODEL)

save_vector_store(PARAGRAPH_VECTOR_STORE_PATH, paragraph_records)
save_vector_store(SECTION_VECTOR_STORE_PATH, section_records)

print(f"Saved paragraph-title embeddings: {len(paragraph_records)} -> {PARAGRAPH_VECTOR_STORE_PATH}")
print(f"Saved section-title embeddings: {len(section_records)} -> {SECTION_VECTOR_STORE_PATH}")

# Example retrievals (kept separate)
paragraph_hits = retrieve_similar_titles(
    PARAGRAPH_VECTOR_STORE_PATH,
    query_text="core network architecture",
    model=EMBEDDING_MODEL,
    top_k=5,
    )
section_hits = retrieve_similar_titles(
    SECTION_VECTOR_STORE_PATH,
    query_text="security procedures",
    model=EMBEDDING_MODEL,
    top_k=5,
    )

print("\nTop paragraph-title matches:")
for hit in paragraph_hits:
    print(f"- {hit['name']} (score={hit['score']:.4f})")

print("\nTop section-title matches:")
for hit in section_hits:
    print(f"- {hit['name']} (score={hit['score']:.4f})")

Paragraph titles: 18
Section titles: 4
Saved paragraph-title embeddings: 18 -> paragraph_title_vectors.json
Saved section-title embeddings: 4 -> section_title_vectors.json

Top paragraph-title matches:
- Stage 6: Deployment (score=0.5237)
- • Model Deployment and Hosting: (score=0.5220)
- Introduction (score=0.5134)
- 2. LLM Agents Using AWS SageMaker JumpStart Foundation Models (score=0.5040)
- Stage 2: Model Initialisation (score=0.4989)

Top section-title matches:
- 4.1     Steps Involved in Model Initialisation (score=0.4735)
- 3 https://huggingface.co/datasets/allenai/wildguardmix (score=0.4525)
- 6.4.2    Comparison between HFT and LoRA (score=0.4103)
- 2 https://ffmpeg.org/ffmpeg.html (score=0.3856)


In [14]:
# Query -> retrieve only paragraph titles
def report_paragraph_titles(query_text: str, top_k: int = 5):
    hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=top_k,
    )

    print(f"Query: {query_text}")
    print("Top paragraph-title matches:")
    if not hits:
        print("No paragraph titles found. Run the embedding build cell first.")
        return

    for i, hit in enumerate(hits, start=1):
        print(f"{i}. {hit['name']} (score={hit['score']:.4f})")

# Edit this query each time you want a new retrieval
user_query = "LLM Fine tuning"
report_paragraph_titles(user_query, top_k=10)

Query: LLM Fine tuning
Top paragraph-title matches:
1. 2. LLM Agents Using AWS SageMaker JumpStart Foundation Models (score=0.6822)
2. Table 6.3: Comparative Analysis of Half Fine-Tuning (HFT) and Low-Rank Adaptation (LoRA). (score=0.6700)
3. • Factual Errors: Outdated information can cause LLMs to provide inaccurate responses. (score=0.6369)
4. • Supervised Fine-tuning Loss (SFT): (score=0.6334)
5. Stage 2: Model Initialisation (score=0.6204)
6. Table 7.1: Detailed Overview of Benchmark Datasets Used for Evaluating Language Model Performance. (score=0.5798)
7. Other tunable parameters include dropout rate, weight decay, and warmup steps. (score=0.5748)
8. Stage 3: Training Setup (score=0.5714)
9. Stage 5: Evaluation and Validation (score=0.5554)
10. • Model Deployment and Hosting: (score=0.5520)


In [15]:
# Retrieve top-3 paragraph titles, then fetch directly connected neighbors in Neo4j
def report_top3_with_graph_neighbors(query_text: str, top_k: int = 3, neighbors_per_title: int = 20):
    top_hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=top_k,
    )

    print(f"Query: {query_text}")
    if not top_hits:
        print("No paragraph-title hits found. Build vector stores first.")
        return

    print("Top paragraph-title hits:")
    for i, hit in enumerate(top_hits, start=1):
        print(f"{i}. {hit['name']} (score={hit['score']:.4f})")

    with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            for i, hit in enumerate(top_hits, start=1):
                title = hit["name"]
                print(f"\nNeighbors for result {i}: {title}")

                rows = list(session.run(
                    """
                    MATCH (p:Paragraph {name: $title})
                    OPTIONAL MATCH (p)-[r]-(n)
                    RETURN
                    p.name AS paragraph_name,
                    type(r) AS rel_type,
                    labels(n) AS neighbor_labels,
                    coalesce(n.name, n.title, toString(id(n))) AS neighbor_name
                    LIMIT $limit
                    """,
                    title=title,
                    limit=neighbors_per_title,
                ))

                if not rows:
                    print("  Paragraph node not found in graph.")
                    continue

                has_neighbors = False
                for row in rows:
                    if row["rel_type"] is None or row["neighbor_name"] is None:
                        continue
                    has_neighbors = True
                    lbl = ":".join(row["neighbor_labels"]) if row["neighbor_labels"] else "Unknown"
                    print(f"  - [{row['rel_type']}] {row['neighbor_name']} ({lbl})")

                if not has_neighbors:
                    print("  Paragraph found, but no directly connected neighbors.")

# Edit query and run this cell
graph_query = user_query if 'user_query' in globals() else "security architecture in 5G core"
report_top3_with_graph_neighbors(graph_query, top_k=3, neighbors_per_title=25)

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=8, column=56, offset=322>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 322, 'line': 8, 'column': 56}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    MATCH (p:Paragraph {name: $title})\n                    OPTIONAL MATCH (p)-[r]-(n)\n                    RETURN\n                    p.name AS paragraph_name,\n                    type(r) AS rel_type,\n                    labels(n) AS neighbor_labels,\n                    coalesce(n.name, n.title, toString(id(n))) AS ne

Query: LLM Fine tuning
Top paragraph-title hits:
1. 2. LLM Agents Using AWS SageMaker JumpStart Foundation Models (score=0.6822)
2. Table 6.3: Comparative Analysis of Half Fine-Tuning (HFT) and Low-Rank Adaptation (LoRA). (score=0.6700)
3. • Factual Errors: Outdated information can cause LLMs to provide inaccurate responses. (score=0.6369)

Neighbors for result 1: 2. LLM Agents Using AWS SageMaker JumpStart Foundation Models
  - [MENTIONED_IN] Entity: Utilising AWS tools -> to deploy these models into applications efficiently and securely (Entity)
  - [MENTIONED_IN] Entity: Amazon Bedrock’s serverless architecture -> allows for -> quick deployment, seamless integration, and secure customisation of FMs (Entity)
  - [MENTIONED_IN] Entity: Amazon Bedrock -> enables the creation of -> intelligent agents that leverage enterprise data and systems (Entity)
  - [MENTIONED_IN] Entity: Amazon Bedrock -> offers -> extensive capabilities for developing secure, private, and responsible generative A

In [18]:
# For each of the top-3 paragraph titles, retrieve first 3 neighbors and map them to section chunks
def build_section_chunk_map(structured_data):
    section_chunks = {}
    current_section = None
    buffer_lines = []

    for item in structured_data:
        item_type = item.get("type")
        text = item.get("text", "").strip()
        if not text:
            continue

        if item_type == "Title" and classify_title_kind(text) == "section":
            if current_section is not None:
                section_chunks[current_section] = "\n".join(buffer_lines).strip()
            current_section = text
            buffer_lines = []
            continue

        if current_section is not None and item_type in {"Title", "Paragraph"}:
            buffer_lines.append(text)

    if current_section is not None:
        section_chunks[current_section] = "\n".join(buffer_lines).strip()

    return section_chunks

def neighbors_per_paragraph_hit(query_text: str, paragraph_top_k: int = 3, neighbors_per_title: int = 3):
    paragraph_hits = retrieve_similar_titles(
        PARAGRAPH_VECTOR_STORE_PATH,
        query_text=query_text,
        model=EMBEDDING_MODEL,
        top_k=paragraph_top_k,
    )

    grouped = []
    with GraphDatabase.driver(URI, auth=NEO4J_AUTH) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            for hit in paragraph_hits:
                rows = session.run(
                    """
                    MATCH (p:Paragraph {name: $title})-[r]-(n)
                    RETURN coalesce(n.name, elementId(n)) AS neighbor_name
                    LIMIT $limit
                    """,
                    title=hit["name"],
                    limit=neighbors_per_title,
                )
                neighbors = []
                seen = set()
                for row in rows:
                    name = row["neighbor_name"]
                    if not name or name in seen:
                        continue
                    seen.add(name)
                    neighbors.append(name)

                grouped.append({
                    "paragraph_title": hit["name"],
                    "paragraph_score": hit["score"],
                    "neighbors": neighbors,
                })

    return grouped

def neighbor_to_section_chunks(query_text: str, paragraph_top_k: int = 3, neighbors_per_title: int = 3, section_top_k: int = 1):
    divider_main = "=" * 100
    divider_sub = "-" * 100
    divider_chunk = "." * 100

    section_chunk_map = build_section_chunk_map(data)
    grouped_hits = neighbors_per_paragraph_hit(
        query_text=query_text,
        paragraph_top_k=paragraph_top_k,
        neighbors_per_title=neighbors_per_title,
    )

    print(divider_main)
    print("PHASE 1 - INPUT QUERY")
    print(divider_main)
    print(f"Query: {query_text}")

    if not grouped_hits:
        print(divider_sub)
        print("No paragraph hits found.")
        return

    print("\n" + divider_main)
    print("PHASE 2 - TOP PARAGRAPH TITLES + FIRST 3 NEIGHBORS EACH")
    print(divider_main)
    for i, g in enumerate(grouped_hits, start=1):
        print(f"{i}. {g['paragraph_title']} (score={g['paragraph_score']:.4f})")
        if g["neighbors"]:
            for j, n in enumerate(g["neighbors"], start=1):
                print(f"   {j}) {n}")
        else:
            print("   No neighbors found for this title.")
        print(divider_sub)

    print("\n" + divider_main)
    print("PHASE 3 - SECTION RETRIEVAL + FULL CHUNK FOR EACH NEIGHBOR")
    print(divider_main)
    for i, g in enumerate(grouped_hits, start=1):
        print(f"\nParagraph {i}: {g['paragraph_title']}")
        print(divider_sub)
        if not g["neighbors"]:
            print("Skipped: no neighbors.")
            continue

        for neighbor in g["neighbors"]:
            print(f"Neighbor query: {neighbor}")
            section_hits = retrieve_similar_titles(
                SECTION_VECTOR_STORE_PATH,
                query_text=neighbor,
                model=EMBEDDING_MODEL,
                top_k=section_top_k,
            )

            if not section_hits:
                print("  No section hits found.")
                print(divider_chunk)
                continue

            for hit in section_hits:
                section_title = hit["name"]
                chunk = section_chunk_map.get(section_title, "")
                print(f"  Matched section: {section_title} (score={hit['score']:.4f})")
                if chunk:
                    print("  Chunk:")
                    print(chunk)
                else:
                    print("  Chunk not found in parsed data.")
                print(divider_chunk)

# Edit query and run
neighbor_section_query = graph_query if 'graph_query' in globals() else "LLM Fine tuning"
neighbor_to_section_chunks(
    query_text=neighbor_section_query,
    paragraph_top_k=3,
    neighbors_per_title=3,
    section_top_k=1,
    )

PHASE 1 - INPUT QUERY
Query: LLM Fine tuning

PHASE 2 - TOP PARAGRAPH TITLES + FIRST 3 NEIGHBORS EACH
1. 2. LLM Agents Using AWS SageMaker JumpStart Foundation Models (score=0.6822)
   1) Entity: Utilising AWS tools -> to deploy these models into applications efficiently and securely
   2) Entity: Amazon Bedrock’s serverless architecture -> allows for -> quick deployment, seamless integration, and secure customisation of FMs
   3) Entity: Amazon Bedrock -> enables the creation of -> intelligent agents that leverage enterprise data and systems
----------------------------------------------------------------------------------------------------
2. Table 6.3: Comparative Analysis of Half Fine-Tuning (HFT) and Low-Rank Adaptation (LoRA). (score=0.6700)
   1) LLMs
   2) models that fail to generalise effectively
   3) Traditional training methods
----------------------------------------------------------------------------------------------------
3. • Factual Errors: Outdated information can 